# Conformal Prediction para Deteccion de Impago

**Objetivo:** aplicar Conformal Prediction sobre el modelo CatBoost entrenado en el notebook 07
para obtener **conjuntos de prediccion** (prediction sets) con garantias estadisticas de cobertura.

En lugar de una prediccion puntual ("este cliente va a hacer default"), Conformal Prediction
nos dice: "con un 90% de confianza, la clase real de este cliente esta en este conjunto {Fully Paid, Default}".

**Libreria utilizada:** [MAPIE](https://mapie.readthedocs.io/) (Model Agnostic Prediction Intervals Estimation),
parte del ecosistema `scikit-learn-contrib`.

**Metricas de referencia del notebook 07 (CatBoost sobre test):**

| Metrica | CatBoost |
|---|---|
| AUC-ROC | 0.7079 |
| Recall (Default) | 0.63 |
| F1 (Default) | 0.43 |

## PASO 1: Carga de datos preprocesados

In [ ]:
import pickle
import numpy as np
import pandas as pd

# Cargar datos preprocesados y filtrados desde pkl
with open('data/filtered/X_train_filtered.pkl', 'rb') as f:
    X_train_filtered = pickle.load(f)
with open('data/filtered/y_train_filtered.pkl', 'rb') as f:
    y_train = pickle.load(f)
with open('data/filtered/X_test_filtered.pkl', 'rb') as f:
    X_test_filtered = pickle.load(f)
with open('data/filtered/y_test_filtered.pkl', 'rb') as f:
    y_test = pickle.load(f)

y_train_flat = y_train.values.ravel()
y_test_flat = y_test.values.ravel()

print(f"Train: {X_train_filtered.shape}")
print(f"Test:  {X_test_filtered.shape}")

n_negative = np.sum(y_train_flat == 0)
n_positive = np.sum(y_train_flat == 1)
print(f"\nClase mayoritaria (Fully Paid): {n_negative}")
print(f"Clase minoritaria (Default):    {n_positive}")
print(f"Ratio:                          {n_negative/n_positive:.2f}")

## PASO 2: Division train / validacion / calibracion

Para Conformal Prediction necesitamos **tres conjuntos** disjuntos:
1. **Train** (60%): entrenar el modelo CatBoost
2. **Validacion** (20%): early stopping durante el entrenamiento
3. **Calibracion** (20%): calcular los scores de no-conformidad para MAPIE

El conjunto de **test** (ya separado) se usa para evaluar los conjuntos de prediccion.

```
Train original (80,000)
├── Train (48,000)      -> entrenar CatBoost
├── Validacion (16,000) -> early stopping
└── Calibracion (16,000) -> calibrar conformal prediction

Test (20,000)           -> evaluar predicciones y conjuntos
```

In [ ]:
from sklearn.model_selection import train_test_split

# Paso 1: separar calibracion (20%) del resto (80%)
X_train_rest, X_cal, y_train_rest, y_cal = train_test_split(
    X_train_filtered, y_train_flat, test_size=0.2, random_state=42, stratify=y_train_flat
)

# Paso 2: separar train (75% del resto = 60% original) y validacion (25% del resto = 20% original)
X_train_sub, X_val, y_train_sub, y_val = train_test_split(
    X_train_rest, y_train_rest, test_size=0.25, random_state=42, stratify=y_train_rest
)

print(f"Train:       {X_train_sub.shape}  ({len(X_train_sub)/len(X_train_filtered)*100:.0f}%)")
print(f"Validacion:  {X_val.shape}  ({len(X_val)/len(X_train_filtered)*100:.0f}%)")
print(f"Calibracion: {X_cal.shape}  ({len(X_cal)/len(X_train_filtered)*100:.0f}%)")
print(f"Test:        {X_test_filtered.shape}")

## PASO 3: Entrenamiento de CatBoost

Usamos los mismos hiperparametros del notebook 07. La unica diferencia es que
entrenamos sobre el 60% de los datos (en lugar del 80%) para reservar el conjunto
de calibracion.

In [ ]:
from catboost import CatBoostClassifier

catboost_model = CatBoostClassifier(
    iterations=5000,
    learning_rate=0.05,
    depth=6,
    l2_leaf_reg=3,
    min_data_in_leaf=20,
    subsample=0.8,
    colsample_bylevel=0.8,
    auto_class_weights='Balanced',
    random_seed=42,
    verbose=200,
    early_stopping_rounds=50,
    eval_metric='AUC'
)

catboost_model.fit(
    X_train_sub, y_train_sub,
    eval_set=(X_val, y_val),
    verbose=200
)

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score

y_pred = catboost_model.predict(X_test_filtered)
y_prob = catboost_model.predict_proba(X_test_filtered)[:, 1]

print("Evaluacion de CatBoost sobre Test:")
print(classification_report(y_test_flat, y_pred, target_names=["Fully Paid", "Default"]))
print(f"AUC-ROC: {roc_auc_score(y_test_flat, y_prob):.4f}")

## PASO 4: Que es Conformal Prediction?

### El problema de las predicciones puntuales

Los modelos de clasificacion producen una prediccion puntual: "Default" o "Fully Paid".
Incluso cuando proporcionan probabilidades (ej: P(Default) = 0.35), estas probabilidades
**no tienen garantias formales** de ser correctas — como vimos en las curvas de calibracion
del notebook 07.

### La solucion: conjuntos de prediccion con garantias

**Conformal Prediction** transforma la prediccion puntual en un **conjunto de prediccion**
(prediction set) que contiene la clase verdadera con una probabilidad garantizada.

Para un nivel de confianza $1 - \alpha$ (ej: 90% con $\alpha = 0.10$):

$$P(Y_{real} \in C(X)) \geq 1 - \alpha$$

Donde $C(X)$ es el conjunto de prediccion para la observacion $X$.

### Tipos de conjuntos posibles (clasificacion binaria)

| Conjunto | Significado | Accion en banca |
|---|---|---|
| {Fully Paid} | El modelo esta seguro: no va a hacer default | **Aprobar** credito |
| {Default} | El modelo esta seguro: va a hacer default | **Rechazar** credito |
| {Fully Paid, Default} | **Incertidumbre**: el modelo no puede decidir | Requiere **revision manual** |
| {} (vacio) | Caso anomalo, no deberia ocurrir con alpha razonable | Investigar |

### Conformal Prediction Split (el metodo que usamos)

1. **Entrenar** el modelo en el conjunto de train
2. **Calibrar** en un conjunto separado: calcular scores de no-conformidad para cada ejemplo
3. **Predecir** en test: construir conjuntos incluyendo las clases cuyo score supera un umbral (quantil de los scores de calibracion)

La **garantia de cobertura** se cumple sin importar la distribucion de los datos,
siempre que calibracion y test sean intercambiables (i.i.d.).

### Por que Conformal Prediction es valioso para riesgo de credito?

1. **Cuantificacion de incertidumbre**: en vez de aceptar/rechazar a ciegas, identificamos
   los casos donde el modelo **no tiene suficiente informacion** para decidir
2. **Garantias formales**: la cobertura del $(1-\alpha)$% se cumple independientemente del modelo
   subyacente — funciona igual con CatBoost, Random Forest, o una red neuronal
3. **Decision informada**: los casos ambiguos ({Fully Paid, Default}) pueden derivarse a
   **analistas humanos** para revision manual, optimizando el uso de recursos
4. **Regulacion**: en el contexto de **Basilea III/IV**, los reguladores exigen que los bancos
   cuantifiquen la incertidumbre en sus modelos de riesgo

## PASO 5: Conformal Prediction con MAPIE

MAPIE (Model Agnostic Prediction Intervals Estimation) implementa Conformal Prediction
para clasificacion y regresion. En la version 1.3, la clase principal es
`SplitConformalClassifier`, que sigue el flujo:

1. **Crear** el wrapper con el modelo ya entrenado (`prefit=True`)
2. **Conformalize**: calcular scores de no-conformidad sobre el conjunto de calibracion
3. **Predict set**: generar conjuntos de prediccion sobre test

Usamos el score de conformidad **LAC** (Least Ambiguous set-valued Classifier), que es
el metodo por defecto y el mas natural para clasificacion binaria.

> **Nota**: en clasificacion binaria (2 clases), los metodos LAC y APS producen resultados
> practicamente equivalentes. La diferencia real entre ambos se manifiesta en problemas
> **multiclase** (3+ clases), donde APS ofrece mejor cobertura condicional por clase.

In [ ]:
from mapie.classification import SplitConformalClassifier

# Niveles de confianza a evaluar
confidence_levels = [0.80, 0.85, 0.90, 0.95]

mapie = SplitConformalClassifier(
    estimator=catboost_model,
    confidence_level=confidence_levels,
    conformity_score="lac",
    prefit=True          # el modelo ya esta entrenado
)

# Calibrar sobre el conjunto de calibracion (calcula scores de no-conformidad)
mapie.conformalize(X_cal, y_cal)
print("MAPIE (SplitConformalClassifier) calibrado")
print(f"Tamano de calibracion: {len(X_cal)} muestras")
print(f"Niveles de confianza:  {confidence_levels}")

### 5.1 Conjuntos de prediccion a distintos niveles de confianza

Generamos prediction sets para niveles de confianza del 80%, 85%, 90% y 95%.

In [ ]:
y_pred_conf, y_ps = mapie.predict_set(X_test_filtered)
# y_ps shape: (n_samples, n_classes, n_confidence_levels) - boolean array

print(f"Shape de prediction sets: {y_ps.shape}")
print(f"  - {y_ps.shape[0]} muestras de test")
print(f"  - {y_ps.shape[1]} clases (0=Fully Paid, 1=Default)")
print(f"  - {y_ps.shape[2]} niveles de confianza: {confidence_levels}")

In [ ]:
import matplotlib.pyplot as plt

print("=" * 70)
print("ANALISIS DE CONJUNTOS DE PREDICCION")
print("=" * 70)

for i, cl in enumerate(confidence_levels):
    # Tamano del conjunto = numero de clases incluidas
    set_sizes = y_ps[:, :, i].sum(axis=1)

    n_empty = (set_sizes == 0).sum()
    n_single = (set_sizes == 1).sum()
    n_both = (set_sizes == 2).sum()

    # Cobertura empirica: la clase real esta en el conjunto?
    coverage = y_ps[np.arange(len(y_test_flat)), y_test_flat.astype(int), i].mean()

    print(f"\nConfianza {cl*100:.0f}%")
    print(f"  Cobertura empirica: {coverage:.4f}  (objetivo: {cl:.2f})")
    print(f"  Conjuntos vacios {{}}:              {n_empty:>6} ({n_empty/len(y_test_flat)*100:>5.1f}%)")
    print(f"  Conjuntos singleton (1 clase):    {n_single:>6} ({n_single/len(y_test_flat)*100:>5.1f}%)")
    print(f"  Conjuntos ambiguos (2 clases):    {n_both:>6} ({n_both/len(y_test_flat)*100:>5.1f}%)")

### 5.2 Cobertura empirica vs. cobertura teorica

La propiedad fundamental de Conformal Prediction es que la **cobertura empirica**
debe ser **al menos** $1 - \alpha$. Verificamos esto graficamente.

In [ ]:
coverages = []
for i, cl in enumerate(confidence_levels):
    cov = y_ps[np.arange(len(y_test_flat)), y_test_flat.astype(int), i].mean()
    coverages.append(cov)

fig, ax = plt.subplots(figsize=(8, 5))
x = [f"{cl*100:.0f}%" for cl in confidence_levels]
ax.bar(x, coverages, color='#3498db', alpha=0.7, label='Cobertura empirica')
ax.plot(range(len(confidence_levels)), confidence_levels, 'ro--', lw=2, markersize=8, label='Cobertura teorica')
ax.set_xlabel('Nivel de confianza')
ax.set_ylabel('Cobertura')
ax.set_title('Cobertura Empirica vs. Teorica')
ax.legend()
ax.set_ylim(0.7, 1.02)
ax.grid(True, alpha=0.3)

for i_bar, (cov, target) in enumerate(zip(coverages, confidence_levels)):
    ax.annotate(f'{cov:.3f}', (i_bar, cov), textcoords="offset points", xytext=(0, 10), ha='center', fontsize=10)

plt.tight_layout()
plt.show()

### 5.3 Desglose por tipo de conjunto y clase real

Analizamos como se distribuyen los conjuntos de prediccion segun la clase real del solicitante.
Esto nos permite entender si el modelo es mas incierto en una clase que en otra.

In [ ]:
cl_focus = 0.90
idx_cl = confidence_levels.index(cl_focus)

# Conjuntos para confianza 90%
ps = y_ps[:, :, idx_cl]
set_sizes = ps.sum(axis=1)

# Separar por clase real
mask_fp = y_test_flat == 0   # Fully Paid
mask_def = y_test_flat == 1  # Default

print(f"Analisis para confianza {cl_focus*100:.0f}%")
print(f"\n{'':>30} {'Fully Paid':>12} {'Default':>12} {'Total':>12}")
print("-" * 70)

for label, desc in [(0, 'Vacio {}'), (1, 'Singleton'), (2, 'Ambiguo {FP, Def}')]:
    mask_size = set_sizes == label
    n_fp = (mask_size & mask_fp).sum()
    n_def = (mask_size & mask_def).sum()
    n_total = mask_size.sum()
    print(f"  {desc:>28} {n_fp:>12,} {n_def:>12,} {n_total:>12,}")

print("-" * 70)
print(f"  {'Total':>28} {mask_fp.sum():>12,} {mask_def.sum():>12,} {len(y_test_flat):>12,}")

# Cobertura por clase
cov_fp = ps[mask_fp, 0].mean()   # Fully Paid cubierto en clase 0
cov_def = ps[mask_def, 1].mean() # Default cubierto en clase 1
print(f"\nCobertura por clase:")
print(f"  Fully Paid: {cov_fp:.4f}")
print(f"  Default:    {cov_def:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Grafico 1: Distribucion de tamanos de conjuntos por nivel de confianza
set_data = {}
for i, cl in enumerate(confidence_levels):
    sizes = y_ps[:, :, i].sum(axis=1)
    set_data[f'{cl*100:.0f}%'] = [
        (sizes == 0).mean() * 100,
        (sizes == 1).mean() * 100,
        (sizes == 2).mean() * 100
    ]

df_sets = pd.DataFrame(set_data, index=['Vacio', 'Singleton', 'Ambiguo'])
df_sets.T.plot(kind='bar', stacked=True, ax=axes[0],
               color=['#e74c3c', '#2ecc71', '#f39c12'])
axes[0].set_title('Distribucion de conjuntos por nivel de confianza')
axes[0].set_xlabel('Nivel de confianza')
axes[0].set_ylabel('Porcentaje de muestras (%)')
axes[0].legend(title='Tipo de conjunto')
axes[0].tick_params(axis='x', rotation=0)

# Grafico 2: Tamano medio del conjunto por clase real
mean_sizes_fp = []
mean_sizes_def = []
for i, cl in enumerate(confidence_levels):
    sizes = y_ps[:, :, i].sum(axis=1)
    mean_sizes_fp.append(sizes[mask_fp].mean())
    mean_sizes_def.append(sizes[mask_def].mean())

x_pos = np.arange(len(confidence_levels))
width = 0.35
axes[1].bar(x_pos - width/2, mean_sizes_fp, width, label='Fully Paid', color='#3498db')
axes[1].bar(x_pos + width/2, mean_sizes_def, width, label='Default', color='#e74c3c')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels([f'{cl*100:.0f}%' for cl in confidence_levels])
axes[1].set_xlabel('Nivel de confianza')
axes[1].set_ylabel('Tamano medio del conjunto')
axes[1].set_title('Tamano medio del conjunto por clase real')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 5.4 Ejemplos practicos de conjuntos de prediccion

Mostramos ejemplos concretos de cada tipo de conjunto para entender la interpretacion practica.

In [ ]:
idx_90 = confidence_levels.index(0.90)
ps_examples = y_ps[:, :, idx_90]
set_sizes_ex = ps_examples.sum(axis=1)
probs = catboost_model.predict_proba(X_test_filtered)

labels = {0: 'Fully Paid', 1: 'Default'}

for tipo, desc, accion in [
    (1, 'SINGLETON - Decision clara', 'Decidir automaticamente'),
    (2, 'AMBIGUO - Incertidumbre', 'Derivar a revision manual')
]:
    mask = set_sizes_ex == tipo
    if mask.sum() == 0:
        continue
    indices = np.where(mask)[0][:3]

    print(f"\n{'='*60}")
    print(f"  {desc}")
    print(f"  Accion recomendada: {accion}")
    print(f"{'='*60}")

    for idx in indices:
        pred_set = [labels[c] for c in range(2) if ps_examples[idx, c]]
        real = labels[y_test_flat[idx]]
        prob_def = probs[idx, 1]
        covered = y_test_flat[idx] in [c for c in range(2) if ps_examples[idx, c]]
        print(f"\n  Muestra #{idx}:")
        print(f"    P(Default) = {prob_def:.3f}")
        print(f"    Conjunto de prediccion: {{{', '.join(pred_set)}}}")
        print(f"    Clase real: {real}")
        print(f"    Cubierto: {'Si' if covered else 'No'}")

## PASO 6: Impacto en decisiones de credito

Analizamos como Conformal Prediction cambiaria el flujo de decision de credito.
Con un nivel de confianza del 90%, clasificamos las solicitudes en tres categorias:
- **Aprobacion automatica**: conjunto = {Fully Paid}
- **Rechazo automatico**: conjunto = {Default}
- **Revision manual**: conjunto = {Fully Paid, Default} (ambiguo)

In [ ]:
idx_90 = confidence_levels.index(0.90)
ps_decision = y_ps[:, :, idx_90]

# Clasificar decisiones
auto_approve = ps_decision[:, 0] & ~ps_decision[:, 1]  # solo Fully Paid
auto_reject = ~ps_decision[:, 0] & ps_decision[:, 1]   # solo Default
manual_review = ps_decision[:, 0] & ps_decision[:, 1]  # ambos
no_decision = ~ps_decision[:, 0] & ~ps_decision[:, 1]  # vacio

print("=" * 70)
print("FLUJO DE DECISION CON CONFORMAL PREDICTION (confianza 90%)")
print("=" * 70)

for mask, label in [
    (auto_approve, "Aprobacion automatica {Fully Paid}"),
    (auto_reject, "Rechazo automatico {Default}"),
    (manual_review, "Revision manual {FP, Default}"),
    (no_decision, "Sin decision {} (anomalo)")
]:
    n = mask.sum()
    if n == 0:
        print(f"\n  {label}: 0 solicitudes")
        continue
    real_def = y_test_flat[mask].mean()
    print(f"\n  {label}:")
    print(f"    Solicitudes: {n:,} ({n/len(y_test_flat)*100:.1f}%)")
    print(f"    Tasa real de default: {real_def:.3f} ({real_def*100:.1f}%)")

# Beneficio vs prediccion puntual
print(f"\n{'='*70}")
print("BENEFICIO DE CONFORMAL PREDICTION")
print(f"{'='*70}")

# Sin conformal: decision binaria sobre todos
total_errors = (y_pred_conf != y_test_flat).sum()
print(f"\nSin conformal: {len(y_test_flat):,} decisiones automaticas")
print(f"  Errores de clasificacion: {total_errors:,}")

# Con conformal: solo decidimos automaticamente en singletons
singleton_mask = auto_approve | auto_reject
n_auto = singleton_mask.sum()
n_manual = manual_review.sum()
errors_auto = (y_pred_conf[singleton_mask] != y_test_flat[singleton_mask]).sum()
print(f"\nCon conformal: {n_auto:,} decisiones automaticas + {n_manual:,} a revision manual")
print(f"  Errores en decisiones automaticas: {errors_auto:,}")
if total_errors > 0:
    print(f"  Reduccion de errores en auto-decisiones: {(total_errors - errors_auto) / total_errors * 100:.1f}%")

## PASO 7: Comparacion con el enfoque naive (umbral sobre probabilidades)

La pregunta natural es: **para que necesitamos conformal prediction si podemos
construir conjuntos directamente con las probabilidades del modelo?**

El enfoque **naive** seria:
1. Para cada muestra, tomar la clase con mayor probabilidad
2. Si esa probabilidad $\geq$ nivel de confianza deseado → **singleton** (solo esa clase)
3. Si no → **ambiguo** (incluir ambas clases)

Ejemplo con confianza 90%: si $P(\text{Fully Paid}) = 0.92$, el conjunto es {Fully Paid}.
Si $P(\text{Fully Paid}) = 0.85$, el conjunto es {Fully Paid, Default}.

Este enfoque parece razonable, pero **asume que las probabilidades del modelo estan
perfectamente calibradas** — que cuando el modelo dice 90%, realmente acierta el 90%
de las veces. Como vimos en el notebook 07, esto no es cierto.

In [ ]:
# Enfoque naive: construir prediction sets directamente con las probabilidades
probs_test = catboost_model.predict_proba(X_test_filtered)

print("=" * 70)
print("COMPARACION: CONFORMAL PREDICTION vs ENFOQUE NAIVE")
print("=" * 70)

for i, cl in enumerate(confidence_levels):
    # --- Enfoque naive ---
    max_prob = probs_test.max(axis=1)           # probabilidad de la clase mas probable
    max_class = probs_test.argmax(axis=1)       # clase con mayor probabilidad

    # Singleton si max_prob >= nivel de confianza, ambiguo si no
    naive_singleton = max_prob >= cl
    naive_n_single = naive_singleton.sum()
    naive_n_both = (~naive_singleton).sum()

    # Cobertura naive: en singletons, cubrimos si la clase predicha es la real
    # En ambiguos (2 clases), siempre cubrimos
    naive_covered = np.where(naive_singleton, max_class == y_test_flat, True)
    naive_coverage = naive_covered.mean()

    # --- Conformal ---
    conf_sizes = y_ps[:, :, i].sum(axis=1)
    conf_n_single = (conf_sizes == 1).sum()
    conf_n_both = (conf_sizes == 2).sum()
    conf_coverage = y_ps[np.arange(len(y_test_flat)), y_test_flat.astype(int), i].mean()

    print(f"\nConfianza {cl*100:.0f}%")
    print(f"  {'':>25} {'Conformal':>12} {'Naive':>12}")
    print(f"  {'-'*50}")
    print(f"  {'Cobertura empirica':>25} {conf_coverage:>12.4f} {naive_coverage:>12.4f}")
    print(f"  {'Singletons':>25} {conf_n_single:>12,} {naive_n_single:>12,}")
    print(f"  {'Ambiguos (2 clases)':>25} {conf_n_both:>12,} {naive_n_both:>12,}")
    print(f"  {'% automatizable':>25} {conf_n_single/len(y_test_flat)*100:>11.1f}% {naive_n_single/len(y_test_flat)*100:>11.1f}%")

### Interpretacion

El enfoque naive construye conjuntos de prediccion **confiando ciegamente** en las
probabilidades del modelo. Pero como las probabilidades de CatBoost con `class_weight='balanced'`
estan **mal calibradas** (notebook 07), el resultado es previsible:

- **Cobertura**: el enfoque naive no garantiza alcanzar la cobertura objetivo porque las
  probabilidades son poco fiables. Conformal prediction si la alcanza porque usa datos
  de calibracion reales para ajustar los umbrales.
- **Singletons**: el enfoque naive puede producir mas o menos singletons, pero **sin garantia**
  de que esos singletons cubran la clase real al nivel prometido.

La leccion clave: **las probabilidades de un modelo no son confianzas**. Conformal prediction
transforma probabilidades no fiables en conjuntos con garantias formales.

## PASO 8: Conformal Prediction Mondrian (por clase)

### El problema de la cobertura marginal

El conformal prediction estandar garantiza cobertura **marginal**:

$$P(Y \in C(X)) \geq 1 - \alpha$$

Pero esto puede ocultar desbalances entre clases. Si la clase mayoritaria (Fully Paid, 80%)
tiene cobertura del 95% y la minoritaria (Default, 20%) tiene cobertura del 75%, la cobertura
marginal ponderada seria ~91% — cumpliendo el objetivo del 90%, pero **fallando en la clase
que mas nos importa** (Default).

### La solucion: Conformal Mondrian

**Conformal Mondrian** (Vovk, 2003) calcula un **umbral de conformidad separado por cada clase**,
garantizando cobertura **class-conditional**:

$$P(Y \in C(X) \mid Y = k) \geq 1 - \alpha \quad \forall k$$

La implementacion es directa:
1. Calcular los scores de conformidad en calibracion: $s_i = 1 - \hat{p}(y_i | x_i)$
2. **Separar** los scores por clase: $S_0$ (Fully Paid) y $S_1$ (Default)
3. Calcular un **cuantil por clase**: $\hat{q}_k$ = cuantil de $S_k$ al nivel $(1-\alpha)(1 + 1/n_k)$
4. Incluir clase $k$ en el prediction set si $1 - \hat{p}(k|x) \leq \hat{q}_k$

MAPIE 1.3.0 no incluye Mondrian nativamente, asi que lo implementamos a mano.

In [ ]:
def conformal_mondrian_predict(probs_cal, y_cal, probs_test, confidence_levels):
    """
    Conformal Mondrian: un cuantil por clase para cobertura class-conditional.
    
    Scores LAC: s(x, y) = 1 - p_hat(y | x)
    """
    classes = np.unique(y_cal)
    n_test = probs_test.shape[0]
    n_levels = len(confidence_levels)
    
    # Scores de conformidad en calibracion: 1 - p(y_true | x)
    cal_scores = 1 - probs_cal[np.arange(len(y_cal)), y_cal.astype(int)]
    
    # Cuantil por clase
    quantiles = {}  # {class: {cl: quantile_value}}
    for k in classes:
        mask_k = y_cal == k
        scores_k = cal_scores[mask_k]
        n_k = len(scores_k)
        quantiles[k] = {}
        for cl in confidence_levels:
            # Correccion finita: quantile level = cl * (1 + 1/n_k)
            level = min(1.0, cl * (1 + 1 / n_k))
            quantiles[k][cl] = np.quantile(scores_k, level)
    
    # Prediction sets: incluir clase k si 1 - p(k|x) <= q_k
    y_ps_mondrian = np.zeros((n_test, len(classes), n_levels), dtype=bool)
    
    for i_cl, cl in enumerate(confidence_levels):
        for k in classes:
            scores_test_k = 1 - probs_test[:, int(k)]
            y_ps_mondrian[:, int(k), i_cl] = scores_test_k <= quantiles[k][cl]
    
    return y_ps_mondrian, quantiles


# Probabilidades de calibracion y test
probs_cal = catboost_model.predict_proba(X_cal)
probs_test = catboost_model.predict_proba(X_test_filtered)

# Mondrian
y_ps_mondrian, quantiles_mondrian = conformal_mondrian_predict(
    probs_cal, y_cal, probs_test, confidence_levels
)

# Mostrar umbrales por clase
print("Umbrales de conformidad por clase (score = 1 - p(y|x)):")
print(f"\n  {'Confianza':>12} {'q (Fully Paid)':>16} {'q (Default)':>16}")
print(f"  {'-'*46}")
for cl in confidence_levels:
    print(f"  {cl*100:>11.0f}% {quantiles_mondrian[0][cl]:>16.4f} {quantiles_mondrian[1][cl]:>16.4f}")

print("\nInterpretacion: un umbral mas alto = criterio mas laxo = mas facil")
print("entrar en el prediction set. Default tiene umbral mas alto porque")
print("el modelo tiene mas dificultad prediciendo esa clase.")

In [ ]:
# Comparativa: Estandar vs Mondrian
mask_fp_test = y_test_flat == 0
mask_def_test = y_test_flat == 1

print("=" * 75)
print("COMPARACION: CONFORMAL ESTANDAR vs CONFORMAL MONDRIAN")
print("=" * 75)

for i, cl in enumerate(confidence_levels):
    # --- Estandar (MAPIE) ---
    std_cov_global = y_ps[np.arange(len(y_test_flat)), y_test_flat.astype(int), i].mean()
    std_cov_fp = y_ps[mask_fp_test, 0, i].mean()
    std_cov_def = y_ps[mask_def_test, 1, i].mean()
    std_sizes = y_ps[:, :, i].sum(axis=1)
    std_n_single = (std_sizes == 1).sum()
    std_n_both = (std_sizes == 2).sum()

    # --- Mondrian ---
    mon_cov_global = y_ps_mondrian[np.arange(len(y_test_flat)), y_test_flat.astype(int), i].mean()
    mon_cov_fp = y_ps_mondrian[mask_fp_test, 0, i].mean()
    mon_cov_def = y_ps_mondrian[mask_def_test, 1, i].mean()
    mon_sizes = y_ps_mondrian[:, :, i].sum(axis=1)
    mon_n_single = (mon_sizes == 1).sum()
    mon_n_both = (mon_sizes == 2).sum()

    print(f"\nConfianza {cl*100:.0f}%")
    print(f"  {'':>28} {'Estandar':>12} {'Mondrian':>12}")
    print(f"  {'-'*54}")
    print(f"  {'Cobertura global':>28} {std_cov_global:>12.4f} {mon_cov_global:>12.4f}")
    print(f"  {'Cobertura Fully Paid':>28} {std_cov_fp:>12.4f} {mon_cov_fp:>12.4f}")
    print(f"  {'Cobertura Default':>28} {std_cov_def:>12.4f} {mon_cov_def:>12.4f}")
    print(f"  {'Diferencia cob. entre clases':>28} {abs(std_cov_fp - std_cov_def):>12.4f} {abs(mon_cov_fp - mon_cov_def):>12.4f}")
    print(f"  {'Singletons':>28} {std_n_single:>12,} {mon_n_single:>12,}")
    print(f"  {'Ambiguos (2 clases)':>28} {std_n_both:>12,} {mon_n_both:>12,}")

In [ ]:
# Grafico comparativo de cobertura por clase
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (method_name, y_ps_method) in zip(axes, [
    ('Estandar (MAPIE)', y_ps),
    ('Mondrian (por clase)', y_ps_mondrian)
]):
    cov_fp_list, cov_def_list = [], []
    for i, cl in enumerate(confidence_levels):
        cov_fp_list.append(y_ps_method[mask_fp_test, 0, i].mean())
        cov_def_list.append(y_ps_method[mask_def_test, 1, i].mean())

    x_pos = np.arange(len(confidence_levels))
    width = 0.3
    bars_fp = ax.bar(x_pos - width/2, cov_fp_list, width, label='Fully Paid', color='#3498db')
    bars_def = ax.bar(x_pos + width/2, cov_def_list, width, label='Default', color='#e74c3c')

    # Lineas de cobertura objetivo
    for cl in confidence_levels:
        ax.axhline(y=cl, color='gray', linestyle='--', alpha=0.3)

    # Anotaciones
    for bar, val in zip(bars_fp, cov_fp_list):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.005, f'{val:.3f}',
                ha='center', va='bottom', fontsize=8)
    for bar, val in zip(bars_def, cov_def_list):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.005, f'{val:.3f}',
                ha='center', va='bottom', fontsize=8)

    ax.set_xticks(x_pos)
    ax.set_xticklabels([f'{cl*100:.0f}%' for cl in confidence_levels])
    ax.set_xlabel('Nivel de confianza')
    ax.set_ylabel('Cobertura por clase')
    ax.set_title(f'Cobertura condicional - {method_name}')
    ax.legend()
    ax.set_ylim(0.65, 1.05)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## PASO 9: Simulacion de caso real — 80% vs 95% de confianza

Simulamos un escenario de produccion donde el banco debe decidir sobre cada
solicitud de credito del conjunto de test. Comparamos dos politicas:

- **Politica conservadora (95%)**: mas seguridad, pero mas casos derivados a especialistas
- **Politica agresiva (80%)**: mas automatizacion, pero menor garantia de cobertura

Para cada politica medimos:
1. **Cobertura**: en que porcentaje de casos la clase real esta dentro del prediction set
2. **Estrechez del set (tasa de singletons)**: en que porcentaje de casos el modelo da una
   respuesta unica (singleton) vs derivar a un especialista (ambiguo)
3. **Calidad de las decisiones automaticas**: para los singletons (donde el modelo decide
   solo), calculamos accuracy, precision y recall — esto mide **como de bueno es el modelo
   cuando se atreve a decidir**

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Niveles de confianza a comparar
politicas = {
    '80% (agresiva)': 0.80,
    '95% (conservadora)': 0.95,
}

print("=" * 75)
print("SIMULACION DE CASO REAL: POLITICA 80% vs 95%")
print("=" * 75)
print(f"\nTotal solicitudes en test: {len(y_test_flat):,}")
print(f"Tasa real de default:      {y_test_flat.mean():.3f} ({y_test_flat.mean()*100:.1f}%)")

resultados_politica = {}

for nombre, cl in politicas.items():
    idx_cl = confidence_levels.index(cl)
    ps = y_ps[:, :, idx_cl]  # (n_samples, 2) boolean
    set_sizes = ps.sum(axis=1)

    # --- 1. Cobertura ---
    covered = ps[np.arange(len(y_test_flat)), y_test_flat.astype(int)]
    coverage = covered.mean()

    # --- 2. Estrechez del set ---
    n_empty = (set_sizes == 0).sum()
    n_singleton = (set_sizes == 1).sum()
    n_ambiguous = (set_sizes == 2).sum()

    # --- 3. Metricas sobre singletons ---
    # En singletons, la prediccion es la unica clase del set
    mask_single = set_sizes == 1
    # La clase predicha es: 1 si {Default}, 0 si {Fully Paid}
    y_pred_single = ps[mask_single, 1].astype(int)  # 1 si Default esta en el set
    y_true_single = y_test_flat[mask_single]

    acc = accuracy_score(y_true_single, y_pred_single)
    prec = precision_score(y_true_single, y_pred_single, zero_division=0)
    rec = recall_score(y_true_single, y_pred_single, zero_division=0)
    f1 = f1_score(y_true_single, y_pred_single, zero_division=0)

    resultados_politica[nombre] = {
        'cl': cl, 'coverage': coverage,
        'n_singleton': n_singleton, 'n_ambiguous': n_ambiguous, 'n_empty': n_empty,
        'acc': acc, 'prec': prec, 'rec': rec, 'f1': f1,
        'y_pred_single': y_pred_single, 'y_true_single': y_true_single,
        'mask_single': mask_single,
    }

    print(f"\n{'─'*75}")
    print(f"  POLITICA: {nombre}")
    print(f"{'─'*75}")
    print(f"\n  1) COBERTURA (clase real dentro del prediction set):")
    print(f"     {coverage:.4f}  ({coverage*100:.1f}% de los casos)")
    print(f"     Objetivo: >= {cl:.2f} | {'CUMPLE' if coverage >= cl else 'NO CUMPLE'}")
    print(f"\n  2) ESTRECHEZ DEL SET (automatizacion vs derivacion):")
    print(f"     Singletons (decision automatica): {n_singleton:>6,} ({n_singleton/len(y_test_flat)*100:>5.1f}%)")
    print(f"     Ambiguos (derivar a especialista): {n_ambiguous:>5,} ({n_ambiguous/len(y_test_flat)*100:>5.1f}%)")
    if n_empty > 0:
        print(f"     Vacios (anomalo):                 {n_empty:>6,} ({n_empty/len(y_test_flat)*100:>5.1f}%)")
    print(f"\n  3) CALIDAD DE LAS DECISIONES AUTOMATICAS (solo singletons):")
    print(f"     Accuracy:   {acc:.4f}  (de {n_singleton:,} decisiones, {int(acc*n_singleton):,} son correctas)")
    print(f"     Precision:  {prec:.4f}  (de los que predice Default, cuantos lo son realmente)")
    print(f"     Recall:     {rec:.4f}  (de los Default reales en singletons, cuantos detecta)")
    print(f"     F1-Score:   {f1:.4f}")

### 9.1 Comparativa directa: 80% vs 95%

Visualizamos lado a lado las dos politicas para entender el trade-off
entre automatizacion y seguridad.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Baseline: CatBoost sin CP, decide sobre TODO el test
y_pred_all = catboost_model.predict(X_test_filtered).astype(int)
acc_all = accuracy_score(y_test_flat, y_pred_all)
prec_all = precision_score(y_test_flat, y_pred_all, zero_division=0)
rec_all = recall_score(y_test_flat, y_pred_all, zero_division=0)
f1_all = f1_score(y_test_flat, y_pred_all, zero_division=0)
errors_all = (y_pred_all != y_test_flat).sum()

r80 = resultados_politica['80% (agresiva)']
r95 = resultados_politica['95% (conservadora)']
N = len(y_test_flat)

print("=" * 80)
print("COMPARATIVA: SIN CP vs CP 80% vs CP 95%")
print("=" * 80)

print(f"\n  {'':>35} {'Sin CP':>12} {'CP 80%':>12} {'CP 95%':>12}")
print(f"  {'':>35} {'(todo test)':>12} {'(singletons)':>12} {'(singletons)':>12}")
print(f"  {'─'*75}")
print(f"  {'Decide sobre':>35} {N:>11,} {r80['n_singleton']:>11,} {r95['n_singleton']:>11,}")
print(f"  {'Deriva a especialista':>35} {'0':>12} {r80['n_ambiguous']:>11,} {r95['n_ambiguous']:>11,}")
print(f"  {'─'*75}")
print(f"  {'Accuracy':>35} {acc_all:>12.4f} {r80['acc']:>12.4f} {r95['acc']:>12.4f}")
print(f"  {'Precision (Default)':>35} {prec_all:>12.4f} {r80['prec']:>12.4f} {r95['prec']:>12.4f}")
print(f"  {'Recall (Default)':>35} {rec_all:>12.4f} {r80['rec']:>12.4f} {r95['rec']:>12.4f}")
print(f"  {'F1 (Default)':>35} {f1_all:>12.4f} {r80['f1']:>12.4f} {r95['f1']:>12.4f}")
print(f"  {'─'*75}")
errors_80 = int((1 - r80['acc']) * r80['n_singleton'])
errors_95 = int((1 - r95['acc']) * r95['n_singleton'])
print(f"  {'Errores en auto-decisiones':>35} {errors_all:>12,} {errors_80:>12,} {errors_95:>12,}")

print(f"\n  Las metricas sobre singletons son mejores porque son los casos faciles.")
print(f"  No es que el modelo mejore — es que solo decide donde esta seguro.")
print(f"  Pero eso es EXACTAMENTE lo util: menos errores en lo que decides solo,")
print(f"  y los casos dificiles van a un humano.")

### 9.1b Que aporta realmente Conformal Prediction?

**Atencion:** comparar accuracy/precision/recall de los singletons vs el modelo completo
es **comparar subconjuntos distintos**. Los singletons son los casos faciles — es normal
que las metricas sean mejores ahi. Es como un estudiante que solo se presenta a los
examenes que sabe: claro que saca mejor nota.

**El valor real de Conformal Prediction NO es mejorar metricas por cherry-picking.**
El valor es **operativo**:

| Sin CP | Con CP |
|---|---|
| El modelo decide sobre **todas** las solicitudes | El modelo decide solo donde esta seguro |
| Se equivoca en X casos y **no sabes cuales** | Se equivoca en menos, y los dificiles los **identifica** |
| No hay mecanismo para separar faciles de dificiles | Tienes un **criterio formal** para derivar |

Sin CP, el banco aprueba/rechaza todas las solicitudes automaticamente y asume todos los
errores. Con CP, las solicitudes dificiles (donde el modelo no puede decidir) se derivan
a un analista humano. **Reduces los errores en las decisiones automaticas**, no porque
el modelo sea mejor, sino porque solo lo dejas decidir donde esta seguro.

La metrica que importa es: **cuantos errores cometes en las decisiones automaticas?**

In [ ]:
# Visualizar la comparativa
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Panel 1: Metricas sobre las decisiones automaticas ---
ax = axes[0]
metrics_names = ['Accuracy', 'Precision\n(Default)', 'Recall\n(Default)', 'F1\n(Default)']
vals_none = [acc_all, prec_all, rec_all, f1_all]
vals_80 = [r80['acc'], r80['prec'], r80['rec'], r80['f1']]
vals_95 = [r95['acc'], r95['prec'], r95['rec'], r95['f1']]

x = np.arange(len(metrics_names))
w = 0.25
b1 = ax.bar(x - w, vals_none, w, color='#95a5a6', alpha=0.8, label=f'Sin CP ({N:,})')
b2 = ax.bar(x,     vals_80,   w, color='#3498db', alpha=0.8, label=f'CP 80% ({r80["n_singleton"]:,})')
b3 = ax.bar(x + w, vals_95,   w, color='#e74c3c', alpha=0.8, label=f'CP 95% ({r95["n_singleton"]:,})')

for bars in [b1, b2, b3]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.008, f'{h:.3f}',
                ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(metrics_names)
ax.set_ylabel('Valor')
ax.set_title('Metricas sobre decisiones automaticas\n(singletons son un subconjunto mas facil)')
ax.legend(fontsize=9)
ax.set_ylim(0, 1.1)
ax.grid(True, alpha=0.3, axis='y')

# --- Panel 2: Errores absolutos en decisiones automaticas ---
ax = axes[1]
errors_80 = int((1 - r80['acc']) * r80['n_singleton'])
errors_95 = int((1 - r95['acc']) * r95['n_singleton'])
derivados_80 = r80['n_ambiguous']
derivados_95 = r95['n_ambiguous']

labels_bar = ['Sin CP', 'CP 80%', 'CP 95%']
errores = [errors_all, errors_80, errors_95]
derivados = [0, derivados_80, derivados_95]
colores = ['#95a5a6', '#3498db', '#e74c3c']

bars_err = ax.bar(labels_bar, errores, color=colores, alpha=0.8, label='Errores automaticos')
bars_der = ax.bar(labels_bar, derivados, bottom=errores, color=colores, alpha=0.3,
                  hatch='//', label='Derivados a humano')

for bar, val in zip(bars_err, errores):
    ax.text(bar.get_x() + bar.get_width()/2, val/2, f'{val:,}\nerrores',
            ha='center', va='center', fontsize=10, fontweight='bold')
for bar, val, bot in zip(bars_der, derivados, errores):
    if val > 0:
        ax.text(bar.get_x() + bar.get_width()/2, bot + val/2, f'{val:,}\na humano',
                ha='center', va='center', fontsize=9)

ax.set_ylabel('Numero de solicitudes')
ax.set_title('Errores en decisiones automaticas\n+ casos derivados a humano')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Conformal Prediction: que aporta realmente?', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### 9.2 Classification report detallado sobre singletons

Mostramos el classification report completo (por clase) calculado **solo** sobre
las muestras donde el modelo da una respuesta unica (singleton). Esto es equivalente
a preguntar: *"cuando el modelo se atreve a decidir, como de bien lo hace?"*

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

for nombre, cl in politicas.items():
    r = resultados_politica[nombre]
    n_s = r['n_singleton']
    n_a = r['n_ambiguous']

    print(f"\n{'='*70}")
    print(f"  POLITICA {nombre}")
    print(f"  Singletons: {n_s:,} decisiones automaticas | Derivados: {n_a:,} a especialistas")
    print(f"{'='*70}")
    print()
    print(classification_report(
        r['y_true_single'], r['y_pred_single'],
        target_names=['Fully Paid', 'Default'],
        digits=4
    ))

    # Matriz de confusion
    cm = confusion_matrix(r['y_true_single'], r['y_pred_single'])
    print(f"  Matriz de confusion (solo singletons):")
    print(f"                    Pred FP   Pred Def")
    print(f"    Real FP:       {cm[0,0]:>7,}    {cm[0,1]:>7,}")
    print(f"    Real Default:  {cm[1,0]:>7,}    {cm[1,1]:>7,}")

### 9.3 Interpretacion: el valor real de Conformal Prediction

**Lo que CP NO hace:**
- No mejora el modelo subyacente
- No hace que precision o recall sean "mejores" en un sentido justo
  (las metricas sobre singletons son sobre un subconjunto mas facil)

**Lo que CP SI hace:**
- Proporciona un **criterio formal y calibrado** para separar los casos donde el
  modelo sabe de los que no sabe. Sin CP, no tienes este mecanismo con garantias.
- **Reduce errores en decisiones automaticas**: los casos que se deciden solos
  tienen menos errores que si decidieras todo automaticamente.
- **Identifica incertidumbre**: los ambiguos son exactamente los casos dificiles
  que deberian ir a revision humana.
- **Garantias formales**: la cobertura se cumple sin importar el modelo ni los datos
  (solo requiere que calibracion y test sean i.i.d.).

**El trade-off 80% vs 95%** es operativo, no de calidad del modelo:
- 80%: mas automatizacion, mas errores automaticos, menos coste de analistas
- 95%: menos automatizacion, menos errores automaticos, mas coste de analistas

**En banca**, donde un error de prediccion puede costar miles de euros (prestar a quien
no va a pagar), tener un mecanismo formal para derivar los casos inciertos a un humano
no es un lujo — es una necesidad regulatoria (Basilea III/IV) y de negocio.

## Conclusiones

### Resultados principales

1. **Conformal Prediction funciona**: la cobertura empirica cumple la garantia teorica
   para todos los niveles de confianza probados.

2. **Identificacion de incertidumbre**: con un 90% de confianza, el modelo identifica
   una proporcion significativa de solicitudes donde **no puede decidir con seguridad**,
   que deberian derivarse a revision manual.

3. **Reduccion de errores**: al separar decisiones automaticas de casos ambiguos,
   se reducen los errores en las decisiones que se toman automaticamente.

4. **El enfoque naive no funciona**: usar directamente las probabilidades del modelo como
   umbrales de confianza produce conjuntos con cobertura distinta a la deseada, porque
   las probabilidades no estan calibradas. Conformal prediction corrige esto.

5. **Mondrian equilibra la cobertura entre clases**: el conformal estandar puede tener
   cobertura desigual entre Fully Paid y Default. Mondrian garantiza cobertura
   class-conditional, a costa de generar mas conjuntos ambiguos.

6. **El trade-off 80% vs 95% es cuantificable**: la simulacion del PASO 9 demuestra que
   al subir la confianza, el modelo se vuelve mas selectivo (menos singletons, mas
   derivaciones), pero las decisiones automaticas que SI toma son de mayor calidad
   (mejor accuracy, precision y recall sobre singletons). La eleccion del nivel de
   confianza depende directamente del apetito de riesgo del banco.

### Estandar vs Mondrian: cual elegir?

- **Estandar**: mas singletons (mas automatizacion), pero la clase minoritaria puede
  tener sub-cobertura. Adecuado si la prioridad es maximizar decisiones automaticas.
- **Mondrian**: cobertura equilibrada entre clases, pero mas casos van a revision manual.
  **Preferible en riesgo de credito**, donde sub-cubrir Default tiene consecuencias graves
  (prestar a quien no va a pagar).

### Implicaciones para banca

- Los casos ambiguos no son un problema, sino una **oportunidad**: permiten focalizar
  el trabajo de los analistas humanos en los casos que realmente lo necesitan
- El nivel de confianza permite ajustar el **trade-off** entre automatizacion y seguridad
- Al evaluar solo los singletons, se obtiene una **medida honesta** de la capacidad del
  modelo: no estamos midiendo sobre casos donde el modelo no sabe — solo donde decide
- Conformal Prediction es **model-agnostic**: se puede aplicar sobre cualquier modelo
  sin modificarlo, lo que facilita su integracion en pipelines existentes